# 机器学习分类项目 - Jupyter Notebook

这个notebook提供了交互式的机器学习分类项目，包含数据探索、模型训练和结果可视化。

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import joblib
import os

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("所有库导入成功！")

## 1. 数据加载和探索

In [ ]:
# 生成示例数据集
from sklearn.datasets import make_classification

print("正在生成示例数据集...")
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=3,
    random_state=42
)

# 创建特征名称
feature_names = [f'feature_{i}' for i in range(X.shape[1])]

# 创建DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

print(f"数据集形状: {df.shape}")
print(f"\n前5行数据:")
display(df.head())
print(f"\n目标类别分布:")
display(df['target'].value_counts())
print(f"\n数据描述统计:")
display(df.describe())

In [ ]:
# 数据可视化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 类别分布
df['target'].value_counts().plot(kind='bar', ax=axes[0, 0], color=['skyblue', 'lightcoral', 'lightgreen'])
axes[0, 0].set_title('目标类别分布')
axes[0, 0].set_xlabel('类别')
axes[0, 0].set_ylabel('数量')

# 特征分布
df.iloc[:, :5].boxplot(ax=axes[0, 1])
axes[0, 1].set_title('前5个特征的分布')
axes[0, 1].set_xticklabels(feature_names[:5], rotation=45)

# 相关性热力图
correlation_matrix = df.iloc[:, :10].corr()
sns.heatmap(correlation_matrix, ax=axes[1, 0], cmap='coolwarm', center=0)
axes[1, 0].set_title('特征相关性热力图（前10个特征）')

# 散点图
scatter = axes[1, 1].scatter(df.iloc[:, 0], df.iloc[:, 1], c=df['target'], cmap='viridis', alpha=0.6)
axes[1, 1].set_title('特征1 vs 特征2（按类别着色）')
axes[1, 1].set_xlabel(feature_names[0])
axes[1, 1].set_ylabel(feature_names[1])
plt.colorbar(scatter, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## 2. 数据预处理

In [ ]:
# 数据预处理
print("正在进行数据预处理...")

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集形状: {X_train_scaled.shape}")
print(f"测试集形状: {X_test_scaled.shape}")
print(f"\n训练集类别分布:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for u, c in zip(unique_train, counts_train):
    print(f"类别 {u}: {c} 个样本 ({c/len(y_train)*100:.1f}%)")

## 3. 模型训练

In [ ]:
# 创建结果字典
results = {}
models = {}

def train_and_evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    """训练和评估模型"""
    print(f"\n正在训练 {model_name}...")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    models[model_name] = model
    results[model_name] = {
        'accuracy': accuracy,
        'predictions': y_pred,
        'model': model
    }
    
    print(f"{model_name} 准确率: {accuracy:.4f}")
    return accuracy

# 训练随机森林
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
train_and_evaluate_model(rf_model, '随机森林', X_train_scaled, y_train, X_test_scaled, y_test)

# 训练逻辑回归
lr_model = LogisticRegression(random_state=42, max_iter=1000, multi_class='multinomial')
train_and_evaluate_model(lr_model, '逻辑回归', X_train_scaled, y_train, X_test_scaled, y_test)

# 训练支持向量机
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
train_and_evaluate_model(svm_model, '支持向量机', X_train_scaled, y_train, X_test_scaled, y_test)

In [ ]:
# 训练神经网络
print("\n正在训练神经网络...")

n_classes = len(np.unique(y_train))

# 构建模型
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(n_classes, activation='softmax')
])

# 编译模型
nn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 训练模型
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# 预测
y_pred_prob = nn_model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_prob, axis=1)

accuracy = accuracy_score(y_test, y_pred)
models['神经网络'] = nn_model
results['神经网络'] = {
    'accuracy': accuracy,
    'predictions': y_pred,
    'model': nn_model,
    'history': history
}

print(f"神经网络 准确率: {accuracy:.4f}")

## 4. 模型评估

In [ ]:
# 模型性能对比
print("模型评估结果")
print("="*50)

for model_name, result in results.items():
    print(f"\n{model_name}:")
    print(f"准确率: {result['accuracy']:.4f}")
    print("分类报告:")
    print(classification_report(y_test, result['predictions']))

In [ ]:
# 可视化结果
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 准确率对比
model_names = list(results.keys())
accuracies = [results[name]['accuracy'] for name in model_names]

bars = axes[0, 0].bar(model_names, accuracies, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
axes[0, 0].set_title('模型准确率对比', fontweight='bold')
axes[0, 0].set_xlabel('模型')
axes[0, 0].set_ylabel('准确率')
axes[0, 0].set_ylim(0, 1)

# 在柱子上显示数值
for bar, acc in zip(bars, accuracies):
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                   f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')

# 混淆矩阵 - 最佳模型
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
best_result = results[best_model_name]

cm = confusion_matrix(y_test, best_result['predictions'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
           xticklabels=['类别0', '类别1', '类别2'],
           yticklabels=['类别0', '类别1', '类别2'])
axes[0, 1].set_title(f'{best_model_name} 混淆矩阵', fontweight='bold')
axes[0, 1].set_xlabel('预测标签')
axes[0, 1].set_ylabel('真实标签')

# 神经网络训练历史
if '神经网络' in results and 'history' in results['神经网络']:
    history = results['神经网络']['history']
    axes[1, 0].plot(history.history['accuracy'], label='训练准确率')
    axes[1, 0].plot(history.history['val_accuracy'], label='验证准确率')
    axes[1, 0].set_title('神经网络训练历史', fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('准确率')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# 特征重要性 - 随机森林
if '随机森林' in models:
    feature_importance = models['随机森林'].feature_importances_
    indices = np.argsort(feature_importance)[-10:]
    
    axes[1, 1].barh(range(len(indices)), feature_importance[indices])
    axes[1, 1].set_yticks(range(len(indices)))
    axes[1, 1].set_yticklabels([feature_names[i] for i in indices])
    axes[1, 1].set_title('随机森林 - 前10个重要特征', fontweight='bold')
    axes[1, 1].set_xlabel('重要性分数')

plt.tight_layout()
plt.show()

## 5. 模型保存和加载

In [ ]:
# 创建模型保存目录
os.makedirs('../models', exist_ok=True)

# 保存传统机器学习模型
for name, model in models.items():
    if name != '神经网络':
        joblib.dump(model, f'../models/{name}.pkl')

# 保存神经网络
if '神经网络' in models:
    models['神经网络'].save('../models/神经网络.h5')

# 保存scaler
joblib.dump(scaler, '../models/scaler.pkl')

print("模型保存完成！")
print(f"保存的模型文件:")
for file in os.listdir('../models'):
    print(f"  - {file}")

In [ ]:
# 模型加载测试
print("正在测试模型加载...")

# 加载随机森林
loaded_rf = joblib.load('../models/随机森林.pkl')
rf_pred = loaded_rf.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"加载的随机森林准确率: {rf_accuracy:.4f}")

# 加载神经网络
from tensorflow.keras.models import load_model
loaded_nn = load_model('../models/神经网络.h5')
nn_pred_prob = loaded_nn.predict(X_test_scaled)
nn_pred = np.argmax(nn_pred_prob, axis=1)
nn_accuracy = accuracy_score(y_test, nn_pred)
print(f"加载的神经网络准确率: {nn_accuracy:.4f}")

print("\n模型加载测试完成！")

## 6. 新数据预测示例

In [ ]:
# 生成新数据进行预测
print("生成新数据进行预测...")

# 生成5个新样本
X_new, y_new = make_classification(
    n_samples=5,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    n_classes=3,
    random_state=123
)

# 预处理新数据
X_new_scaled = scaler.transform(X_new)

print("新数据形状:", X_new_scaled.shape)
print("\n使用不同模型对新数据进行预测:")

for model_name, model in models.items():
    if model_name == '神经网络':
        pred_prob = model.predict(X_new_scaled)
        pred = np.argmax(pred_prob, axis=1)
        confidence = np.max(pred_prob, axis=1)
    else:
        pred = model.predict(X_new_scaled)
        confidence = model.predict_proba(X_new_scaled)
        confidence = np.max(confidence, axis=1)
    
    print(f"\n{model_name}预测结果:")
    for i, (p, c) in enumerate(zip(pred, confidence)):
        print(f"  样本{i+1}: 预测类别={p}, 置信度={c:.3f}")

## 7. 总结

In [ ]:
# 项目总结
print("机器学习分类项目总结")
print("="*50)

# 找出最佳模型
best_model = max(results.keys(), key=lambda x: results[x]['accuracy'])
best_accuracy = results[best_model]['accuracy']

print(f"\n最佳模型: {best_model}")
print(f"最佳准确率: {best_accuracy:.4f}")
print(f"\n所有模型性能:")
for model_name, result in results.items():
    print(f"  {model_name}: {result['accuracy']:.4f}")

print(f"\n项目特点:")
print("✓ 使用了多种机器学习算法进行对比")
print("✓ 包含了传统机器学习和深度学习方法")
print("✓ 提供了完整的数据预处理和可视化")
print("✓ 支持模型保存和加载")
print("✓ 包含新数据预测功能")

print(f"\n项目文件已保存到 '../models/' 目录")
print("你可以使用这些模型来预测新的数据！")